# 1. Prepare GEX data

In [1]:
DROP_NULLS = True
DROP_LOUVEAU = True
SELECT_PRE_TREATMENT = True
SELECT_RNA_SEQ = False

## Some mappings

In [2]:
rna_seq_sources = ['Hugo et al.', 'Kwong et al.', 'Yan et al.']
q_pcr_sources = ['Louveau et al.']
micro_array_sources = ['Long et al.', 'Rizos et al.']

source_map = {
    'doi:10.1016/j.cell.2015.07.061': 'Hugo et al.',
    'doi:10.1172/JCI78954DS1': 'Kwong et al.',
    'doi:10.1158/1078-0432.CCR-18-0720': 'Yan et al.',
    'doi:10.3390/cancers11081203': 'Louveau et al.',
    'doi:10.1038/ncomms6694': 'Long et al.',
    'doi:10.1158/1078-0432.CCR-13-3122': 'Rizos et al.'
}

In [3]:
import polars as pl

gex = pl.read_csv("../dataset/original/gene_expressions.csv")
gex = gex.with_columns(pl.col('source').replace(source_map))
gex

id,creation_datetime,patientID,sample_id,HGNC,GeneID,description,value,temporality,source
i64,str,str,str,str,str,str,f64,str,str
1,"""2025-04-23 23:02:05.503260""","""LM_1""","""LMSAM_1""","""BRAF""",null,null,7.60798,"""pre treatment""","""Louveau et al."""
2,"""2025-04-23 23:02:05.503285""","""LM_1""","""LMSAM_1""","""RAF1""",null,null,16.095204,"""pre treatment""","""Louveau et al."""
3,"""2025-04-23 23:02:05.503298""","""LM_1""","""LMSAM_1""","""ARAF""",null,null,4.1515,"""pre treatment""","""Louveau et al."""
4,"""2025-04-23 23:02:05.503312""","""LM_1""","""LMSAM_1""","""PDGFRB""",null,null,1.199885,"""pre treatment""","""Louveau et al."""
5,"""2025-04-23 23:02:05.503324""","""LM_1""","""LMSAM_1""","""IGF1R""",null,null,5.47246,"""pre treatment""","""Louveau et al."""
…,…,…,…,…,…,…,…,…,…
8641387,"""2025-04-24 00:42:16.629746""","""HL_Shi-40""","""Pt21-DP2""","""ZYG11A""",null,null,0.019542,"""progression""","""Hugo et al."""
8641388,"""2025-04-24 00:42:16.629757""","""HL_Shi-40""","""Pt21-DP2""","""ZYG11B""",null,null,4.04421,"""progression""","""Hugo et al."""
8641389,"""2025-04-24 00:42:16.629768""","""HL_Shi-40""","""Pt21-DP2""","""ZYX""",null,null,85.7967,"""progression""","""Hugo et al."""


## Select only 'pre-treatment', drop Louveau

In [4]:
if SELECT_PRE_TREATMENT == True:
    gex = gex.filter((pl.col('temporality') == 'pre treatment'))
if SELECT_RNA_SEQ == True:
    gex = gex.filter(pl.col('source').is_in(rna_seq_sources))
if DROP_LOUVEAU == True:
    gex = gex.filter(pl.col('source') != 'Louveau et al.')

## Drop useless features

In [5]:
gex = gex.drop(['id', 'creation_datetime', 'GeneID', 'description', 'temporality'])

## Sample is useless if patientID or HGNC is not given

In [6]:
gex.select(pl.all().null_count())

patientID,sample_id,HGNC,value,source
u32,u32,u32,u32,u32
92181,0,207000,0,0


In [7]:
if DROP_NULLS == True:
    gex = gex.drop_nulls(subset=['patientID', 'HGNC'])
    gex.select(pl.all().null_count())
    
gex.select(pl.all().null_count())

patientID,sample_id,HGNC,value,source
u32,u32,u32,u32,u32
0,0,0,0,0


## Add Method column

In [8]:
gex = gex.with_columns(
    pl.when(pl.col('source').is_in(rna_seq_sources))
    .then(pl.lit('RNA-seq'))
    .when(pl.col('source').is_in(micro_array_sources))
    .then(pl.lit('micro-array'))
    .otherwise(pl.lit('qPCR'))
    .alias('Method')
)
gex

patientID,sample_id,HGNC,value,source,Method
str,str,str,f64,str,str
"""YR_5306""","""03660445B""","""NAT2""",0.0,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""ADA""",26.233973,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""CDH2""",1.138609,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""AKT3""",12.692677,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""GAGE12F""",0.0,"""Yan et al.""","""RNA-seq"""
…,…,…,…,…,…
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11A""",0.0625681,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11B""",5.74608,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYX""",45.907933,"""Hugo et al.""","""RNA-seq"""


## Remove duplicate (sample-gene) rows

In [9]:
dupes_pl = (
    gex
    .filter(pl.len().over(['HGNC', 'sample_id']) > 1)
    .sort(['HGNC', 'sample_id'])
)
print(dupes_pl)

shape: (21_042, 6)
┌───────────┬───────────┬────────┬───────┬──────────────┬─────────┐
│ patientID ┆ sample_id ┆ HGNC   ┆ value ┆ source       ┆ Method  │
│ ---       ┆ ---       ┆ ---    ┆ ---   ┆ ---          ┆ ---     │
│ str       ┆ str       ┆ str    ┆ f64   ┆ str          ┆ str     │
╞═══════════╪═══════════╪════════╪═══════╪══════════════╪═════════╡
│ KC_10     ┆ 10A       ┆ ACE    ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_10     ┆ 10A       ┆ ACE    ┆ 2.84  ┆ Kwong et al. ┆ RNA-seq │
│ KC_12     ┆ 12A       ┆ ACE    ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_12     ┆ 12A       ┆ ACE    ┆ 10.27 ┆ Kwong et al. ┆ RNA-seq │
│ KC_13     ┆ 13A       ┆ ACE    ┆ 18.67 ┆ Kwong et al. ┆ RNA-seq │
│ …         ┆ …         ┆ …      ┆ …     ┆ …            ┆ …       │
│ KC_7      ┆ 7A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_7      ┆ 7A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_9      ┆ 9A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_9      ┆ 9A        ┆ mir

In [10]:
gex = gex.sort('value', descending=True).unique(subset=['HGNC', 'sample_id'], keep='first')

dupes_pl = (
    gex
    .filter(pl.len().over(['HGNC', 'sample_id']) > 1)
    .sort(['HGNC', 'sample_id'])
)
print(dupes_pl)

shape: (0, 6)
┌───────────┬───────────┬──────┬───────┬────────┬────────┐
│ patientID ┆ sample_id ┆ HGNC ┆ value ┆ source ┆ Method │
│ ---       ┆ ---       ┆ ---  ┆ ---   ┆ ---    ┆ ---    │
│ str       ┆ str       ┆ str  ┆ f64   ┆ str    ┆ str    │
╞═══════════╪═══════════╪══════╪═══════╪════════╪════════╡
└───────────┴───────────┴──────┴───────┴────────┴────────┘


## Save pre-processed GEX

In [11]:
print(gex)
gex.write_csv(f'../dataset/created/gex.csv')

shape: (4_162_394, 6)
┌────────────┬──────────────────┬───────────┬───────────┬──────────────┬─────────────┐
│ patientID  ┆ sample_id        ┆ HGNC      ┆ value     ┆ source       ┆ Method      │
│ ---        ┆ ---              ┆ ---       ┆ ---       ┆ ---          ┆ ---         │
│ str        ┆ str              ┆ str       ┆ f64       ┆ str          ┆ str         │
╞════════════╪══════════════════╪═══════════╪═══════════╪══════════════╪═════════════╡
│ LR_SMU-020 ┆ 46225 PreC       ┆ PTPN20B   ┆ 9.396869  ┆ Long et al.  ┆ micro-array │
│ KC_19      ┆ 19A              ┆ LOC286189 ┆ 0.0       ┆ Kwong et al. ┆ RNA-seq     │
│ LR_MTP-034 ┆ 28518_ 085G PreC ┆ BRF1      ┆ 95.00407  ┆ Long et al.  ┆ micro-array │
│ YR_2420    ┆ 05320132B        ┆ USP45     ┆ 5.097668  ┆ Yan et al.   ┆ RNA-seq     │
│ LR_MTP-009 ┆ 27551 PreB       ┆ MDM4      ┆ 43.71098  ┆ Long et al.  ┆ micro-array │
│ …          ┆ …                ┆ …         ┆ …         ┆ …            ┆ …           │
│ YR_2220    ┆ 053202

## Create GEX_MAT

In [12]:
gex_mat = gex.pivot(on='sample_id', index='HGNC', values='value')
gex_mat

HGNC,46225 PreC,19A,28518_ 085G PreC,05320132B,27551 PreB,05320425B,05320393B,28178_018A PreB,56025 PreB,78853 PreB,05420211C,28094 PreB,05320108C,2A,05320011B,03660445B,Pt4-baseline,05320384B,05320459B,27473_010F PreB,08766 PreC_014G,10A,12A,28115 PreB,28139_072I PreB,05320353C,28702_010D PreB,27228_035A PreB,9A,Pt6-baseline,30230_077K PreB,25A,Pt3-baseline,Pt1-baseline,05420138C,05420180C,…,05320424B,05320234B,05320102B,8766_022H PreC,05320012B,05320184B,05320436B,05320229B,03660602B,31861 PreC,56460 PreB,85284 PreC,8755_035H PreB,05320119C,05320143B,Pt15-baseline,04240076F,Pt10-baseline,Pt8-baseline,46844 PreB,Pt9-baseline,05320399C,05320191B,Pt2-baseline,05320217B,05320232B,05320093C,56241 PreB,05320141B,03660502B,05320385B,Pt5-baseline,31721 PreC,05320372C,03660555B,04240120C,05320420B
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""PTPN20B""",9.396869,0.03,6.368301,null,3.090367,null,null,10.68724,10.41765,9.279964,null,10.70889,null,0.13,null,null,0.018797,null,null,7.653714,3.804935,1.64,0.06,9.221467,6.558414,null,7.227451,3.758798,0.06,0.0,14.44122,4.93,0.688632,0.067726,null,null,…,null,null,null,3.443724,null,null,null,null,null,9.82723,18.33162,2.309697,0.3953528,null,null,0.048088,null,0.064897,0.0558554,10.0994,0.01292,null,null,0.0,null,null,null,11.24184,null,null,null,0.0,4.931131,null,null,null,null
"""LOC286189""",null,0.0,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,0.0,null,null,null,null,0.0,0.0,null,null,null,null,null,0.0,0.0,null,0.0,0.0,0.0,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,0.023045,0.208488,null,0.0,null,null,0.0,null,null,null,null,null,null,null,0.041442,null,null,null,null,null
"""BRF1""",49.39267,10.71,95.00407,1.795553,173.9156,2.578136,1.600722,60.29753,32.56132,38.63843,1.515172,62.01594,4.923899,7.08,2.024606,1.339246,7.286745,1.524644,2.394089,104.9102,32.16647,13.23,13.1,83.3833,59.47157,2.052539,37.80637,47.35797,13.11,7.430323,54.10693,14.98,7.922765,13.68335,1.83939,1.873611,…,1.371745,2.419735,1.841025,33.30017,1.150895,1.827596,2.673152,2.852874,1.63143,36.03552,92.03206,54.0781,102.6149,3.253092,3.809062,8.60229,1.814024,12.8735,13.17745,38.39656,11.52235,1.866429,10.23718,11.5658,2.809016,1.968942,1.613485,49.02478,2.610634,3.258449,2.003498,20.9621,46.51017,1.622799,2.592774,2.473781,1.933968
"""USP45""",7.081241,0.68,0.2609854,5.097668,-3.486656,3.349943,5.332697,20.30538,9.841049,26.03193,2.896497,-0.198408,2.523963,2.3,5.250066,3.606313,1.37401,5.464629,3.997551,-3.868389,11.48349,4.22,1.58,8.966571,14.02993,2.466697,3.201859,5.020415,1.24,1.92725,3.245033,3.3,2.348565,3.30842,3.237047,3.461676,…,2.771553,3.107156,3.229641,-1.028621,5.954242,7.90313,6.111881,3.385774,5.287425,11.05713,11.85493,7.861904,7.625168,3.781922,5.966933,3.178015,2.597963,0.2084015,2.200025,20.33314,4.81934,4.385063,2.872507,2.6784,4.936217,3.11169,3.711878,13.07029,3.3280727,5.084979,3.120472,0.1929545,18.05258,5.489969,2.299355,5.863929,4.398097
"""MDM4""",11.75806,1.8,57.91211,10.938775,43.71098,7.300348,7.429751,117.8011,24.59747,82.26534,12.258901,101.3182,13.539352,1.53,10.040057,8.918833,4.865465,13.668457,9.945125,46.7431,63.83135,7.43,1.36,89.73165,243.7735,8.97097,53.65295,191.7698,2.64,2.821533,54.38245,3.3,5.757105,22.7889,5.178977,7.46299,…,6.707227,14.561871,15.185525,34.66736,6.214277,15.538027,12.53732,6.493648,10.690125,30.78123,16.57665,25.29574,55.33653,12.411452,17.02688,14.94485,5.328773,1.37434,8.776725,38.22955,10.7715,12.333999,8.588093,3.578945,11.292212,8.374826,8.051375,18.00152,7.618639,13.015214,7.89604,1.30785,30.24246,11.538226,7.184818,7.411554,18.194329
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…

In [13]:
gex_mat.write_csv(f'../dataset/created/gex_mat.csv')